# Diabetes Classification with Logistic Regression

This notebook follows the exact chronology of a Machine Learning project:

1. Understand the problem
2. Load and explore the data
3. Split the data into train/test sets
4. Standardize the data
5. Train a Logistic Regression model
6. Evaluate the model
7. Visualize performance
8. Plot the ROC Curve

Goal: Predict whether a person has diabetes (1) or not (0).


# Exercise 1 - Understanding the problem and Data Collection

Before training a model, we must:
- Load the dataset
- Understand the variables
- Identify the target variable
- Split the data into training and testing sets


In [ ]:
import pandas as pd

df = pd.read_csv("data/diabetes_prediction_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())

print("\nTarget distribution:")
print(df["diabetes"].value_counts())


In [ ]:
# Positive and negative cases

positive_cases = (df["diabetes"] == 1).sum()
negative_cases = (df["diabetes"] == 0).sum()

print("Positive cases:", positive_cases)
print("Negative cases:", negative_cases)


In [ ]:
# Split data into features (X) and target (y)

from sklearn.model_selection import train_test_split

X = df.drop("diabetes", axis=1)
y = df["diabetes"]

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

# Why split?
# Training data is used to teach the model.
# Testing data is used to evaluate the model on unseen data.


# Exercise 2 - Model Picking and Standardization

Why Logistic Regression?

- We have a classification problem.
- Only two possible outputs: diabetes or no diabetes.
- Logistic Regression is simple and effective.
- It predicts probabilities.

Why StandardScaler?

- Numerical variables may have very different scales.
- Standardization helps the model learn more efficiently.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)


# Exercise 3 - Model Training

Now we train the Logistic Regression model.

Remember:
- fit() = learning phase
- predict() = prediction phase


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)

print("Model trained successfully.")


# Exercise 4 - Evaluation Metrics

Important metrics:

- Accuracy → Overall correctness.
- Precision → Among predicted diabetics, how many are truly diabetic?
- Recall → Among real diabetics, how many did we detect?
- F1-score → Balance between Precision and Recall.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall   :", round(recall,4))
print("F1 Score :", round(f1,4))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.bar(
    ["Accuracy","Precision","Recall","F1"],
    [accuracy, precision, recall, f1]
)
plt.title("Evaluation Metrics")
plt.show()


In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()

# Interpretation:
# TN = Correctly predicted non-diabetic.
# TP = Correctly predicted diabetic.
# FP = False alarm.
# FN = Missed diabetic case.


### Comment

A good model generally has:
- High accuracy
- High precision
- High recall
- High F1-score

Precision and Recall often have a trade-off.


# Exercise 5 - Decision Boundary Visualization

A decision boundary can only be displayed in 2D.

We therefore use two numerical features.


In [ ]:
import numpy as np

feat_x = 'HbA1c_level' if 'HbA1c_level' in X.columns else num_cols[0]
feat_y = 'blood_glucose_level' if 'blood_glucose_level' in X.columns else num_cols[1]

from sklearn.pipeline import Pipeline

X2_train = X_train[[feat_x, feat_y]]
X2_test = X_test[[feat_x, feat_y]]

model_2d = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

model_2d.fit(X2_train, y_train)

acc_2d = model_2d.score(X2_test, y_test)

x_min, x_max = X2_train[feat_x].min()-1, X2_train[feat_x].max()+1
y_min, y_max = X2_train[feat_y].min()-1, X2_train[feat_y].max()+1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200)
)

Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8,6))
plt.contourf(xx, yy, Z, alpha=0.3)
plt.scatter(X2_test[feat_x], X2_test[feat_y], c=y_test)
plt.title(f"Decision Boundary - Accuracy={acc_2d:.3f}")
plt.xlabel(feat_x)
plt.ylabel(feat_y)
plt.show()


# Exercise 6 - ROC Curve

ROC Curve evaluates the model across multiple thresholds.

AUC Interpretation:
- 0.5 = random model
- 0.7+ = acceptable
- 0.8+ = good
- 0.9+ = excellent


In [ ]:
from sklearn import metrics

y_proba = clf.predict_proba(X_test)[:,1]

fpr, tpr, _ = metrics.roc_curve(y_test, y_proba)
auc = metrics.roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

print("AUC:", round(auc,4))


## Final Interpretation

If the ROC curve stays well above the diagonal and the AUC is close to 1, the model distinguishes diabetic and non-diabetic patients effectively.

This completes the Machine Learning workflow:
Data → Train/Test Split → Standardization → Training → Evaluation → ROC Curve.
